# Score a document parser against the parsing ground truth

This notebook scores one parser on one set of pages. It takes a folder holding one markdown file per page,
runs the OmniDocBench evaluator on it through the corrected launcher in this folder, and shows the scores:
overall, by document type, page by page, and line by line for any page you choose.

Before the first run, read sections 5 to 8 of `README.md` in the folder above this one. In short, you need:

1. **Your parser's output**, one file per page, named after the page's image: `CASE001_invoice.md` for
   `CASE001_invoice.png`. README section 5 says what the file may hold.
2. **The OmniDocBench evaluator**, at the commit README section 6 names, installed in a Python
   environment.

**Work on a copy of this notebook, outside the kit.** Saving a notebook changes it, and the kit's verifier
reports any changed or added file. Copy `tools/score_parsing.ipynb` to a folder of your own, open the
copy, and set `KIT` below to where the kit is.

Run the cells in order. Only the first code cell needs editing. Nothing is written inside the kit: every
result goes to `WORK_DIR`.

## 1. Settings

Edit this cell, then run it. Every path is yours to set; the comments say what each one is.

In [ ]:
import sys
from pathlib import Path

# The unpacked kit: the folder that holds README.md, MANIFEST.json, parsing/ and tools/.
KIT = Path("/path/to/the/unpacked/kit")

# Which set to score: "plain", "distractor", "labelled_varied", "degraded",
# "typeset_plain" or "typeset_distractor" (README section 3).
SET = "plain"

# Your parser's output for that set: one <image stem>.md per page.
# The evaluator names its result files after this folder, so give each run its own folder name.
PREDICTIONS = Path("/path/to/your/parser/output") / SET

# Where the results are written. Keep it outside the kit, and give each run its own folder.
WORK_DIR = Path.home() / "parsing_scores" / PREDICTIONS.name

# The OmniDocBench evaluator's folder (README section 6).
OMNIDOCBENCH = Path.home() / "OmniDocBench"

# The Python of the environment the evaluator is installed in. The default is this notebook's own
# kernel; if the evaluator lives in another environment, give that environment's python here,
# for example Path("/opt/conda/envs/omnidocbench/bin/python").
EVALUATOR_PYTHON = Path(sys.executable)

# A page with no markdown file is scored as wholly wrong (text and reading order 1, tables 0).
# The preflight stops when a page is missing unless this is True.
SCORE_MISSING_PAGES_AS_WRONG = False

## 2. Functions

Run this cell once. It defines everything the later cells use and changes nothing on disk.

`run_evaluator` writes the evaluator's configuration into the work folder and runs the evaluator there
through `omnidocbench_formula_free.py`, the launcher beside this notebook. The launcher corrects three
faults in how the evaluator reads a parser's output (README section 7); the evaluator's own files are
never changed.

In [ ]:
import hashlib
import html
import json
import os
import re
import subprocess

import pandas as pd

KIT = Path(KIT).resolve()
if not (KIT / "MANIFEST.json").is_file() or not (KIT / "parsing").is_dir():
    raise SystemExit(
        f"KIT is {KIT}, which is not the unpacked kit: it holds no MANIFEST.json and parsing/ folder."
    )
TOOLS = KIT / "tools"
LAUNCHER = TOOLS / "omnidocbench_formula_free.py"
MANIFEST = json.loads((KIT / "MANIFEST.json").read_text())
EVALUATOR_COMMIT = MANIFEST["parsing"]["evaluator"]["commit"]
SETS = sorted(path.stem for path in (KIT / "parsing").glob("*.json"))

# What the evaluator is asked to measure: the same settings the published figures in the README used.
METRICS = {
    "text_block": {"metric": ["Edit_dist"]},
    "table": {"metric": ["TEDS", "Edit_dist"]},
    "reading_order": {"metric": ["Edit_dist"]},
}
HEADLINE = {
    "Text edit distance (lower is better)": ("text_block", "Edit_dist", "ALL_page_avg"),
    "Table TEDS (higher is better)": ("table", "TEDS", "all"),
    "Table TEDS, structure only (higher is better)": ("table", "TEDS_structure_only", "all"),
    "Table edit distance (lower is better)": ("table", "Edit_dist", "ALL_page_avg"),
    "Reading-order edit distance (lower is better)": ("reading_order", "Edit_dist", "ALL_page_avg"),
}
# The evaluator must not fetch anything while it scores; the metrics used load no model.
OFFLINE = {"HF_HUB_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}


def load_pages(set_name):
    """The ground-truth pages of one set, as OmniDocBench pages."""
    if set_name not in SETS:
        raise ValueError(f"SET is {set_name!r}; the kit holds {SETS}.")
    return json.loads((KIT / "parsing" / f"{set_name}.json").read_text())


def stem(page):
    """A page's file name without its extension: the name its markdown file must have."""
    return Path(page["page_info"]["image_path"]).stem


def preflight(pages, predictions, work_dir):
    """Check the settings before anything runs, and say plainly what is wrong."""
    problems = []
    if not (OMNIDOCBENCH / "pdf_validation.py").is_file():
        problems.append(
            f"OMNIDOCBENCH: no pdf_validation.py in {OMNIDOCBENCH}; is it the evaluator's folder?"
        )
    else:
        differing = [
            relative
            for relative, digest in MANIFEST["evaluator_files"].items()
            if not (OMNIDOCBENCH / relative).is_file()
            or hashlib.sha256((OMNIDOCBENCH / relative).read_bytes()).hexdigest() != digest
        ]
        if differing:
            problems.append(
                f"OMNIDOCBENCH is not the evaluator at commit {EVALUATOR_COMMIT}: {len(differing)} of its "
                f"{len(MANIFEST['evaluator_files'])} source files differ or are missing, for example "
                f"{differing[:3]}. Download and unpack it again (README section 6)."
            )
    if not Path(EVALUATOR_PYTHON).is_file():
        problems.append(f"EVALUATOR_PYTHON: {EVALUATOR_PYTHON} is not a file.")
    resolved = Path(work_dir).resolve()
    if resolved == KIT or KIT in resolved.parents:
        problems.append(f"WORK_DIR {resolved} is inside the kit; choose a folder outside it.")
    if not Path(predictions).is_dir():
        problems.append(f"PREDICTIONS: {predictions} is not a folder.")
    else:
        wanted = {stem(page) for page in pages}
        found = {path.stem for path in Path(predictions).glob("*.md")}
        missing, extra = sorted(wanted - found), sorted(found - wanted)
        empty = sorted(s for s in wanted & found if not (Path(predictions) / f"{s}.md").read_text().strip())
        print(f"{len(wanted & found)} of {len(wanted)} pages have a markdown file in {predictions}")
        if extra:
            print(
                f"note: {len(extra)} markdown files match no page of this set and are ignored: {extra[:5]}"
            )
        if empty:
            print(
                f"note: {len(empty)} markdown files are empty and will score as wholly wrong: {empty[:5]}"
            )
        marked = 0
        for name in sorted(wanted & found):
            content = (Path(predictions) / f"{name}.md").read_text()
            cells = re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", content, re.S)
            cells += [
                c for line in content.splitlines() if line.lstrip().startswith("|") for c in line.split("|")
            ]
            marked += sum(bool(re.search(r"<(b|strong|i|em)>|\*\*[^*]+\*\*", cell)) for cell in cells)
        if marked:
            print(
                f"warning: {marked} table cells hold bold or italic markup. Inside a table cell it is compared "
                "as written and lowers the table scores (README section 5): strip it when you export."
            )
        if missing and not SCORE_MISSING_PAGES_AS_WRONG:
            problems.append(
                f"{len(missing)} pages have no markdown file, for example {missing[:5]}. Either write them, "
                "or set SCORE_MISSING_PAGES_AS_WRONG = True to score each as wholly wrong."
            )
        elif missing:
            print(f"note: {len(missing)} pages have no markdown file and will score as wholly wrong.")
    if problems:
        raise SystemExit("Cannot score yet:\n- " + "\n- ".join(problems))
    print("Preflight passed.")


def run_evaluator(set_name, predictions, work_dir):
    """Score one folder of markdown files against one set, and return where the results are.

    The evaluator writes into <work_dir>/result/, naming every file after the predictions folder.
    """
    predictions, work_dir = Path(predictions).resolve(), Path(work_dir).resolve()
    work_dir.mkdir(parents=True, exist_ok=True)
    config = {
        "end2end_eval": {
            "metrics": METRICS,
            "dataset": {
                "dataset_name": "end2end_dataset",
                "ground_truth": {"data_path": str(KIT / "parsing" / f"{set_name}.json")},
                "prediction": {"data_path": str(predictions)},
                "match_method": "quick_match",
            },
        }
    }
    config_file = work_dir / "end2end.json"  # JSON is valid YAML, which is what the evaluator reads
    config_file.write_text(json.dumps(config, indent=2))
    command = [
        str(EVALUATOR_PYTHON),
        str(LAUNCHER),
        "--checkout",
        str(OMNIDOCBENCH),
        "--config",
        str(config_file),
    ]
    done = subprocess.run(
        command, cwd=work_dir, env={**os.environ, **OFFLINE}, capture_output=True, text=True
    )
    (work_dir / "evaluator.log").write_text(done.stdout + "\n--- stderr ---\n" + done.stderr)
    prefix = work_dir / "result" / f"{predictions.name}_quick_match"
    if done.returncode != 0 or not Path(f"{prefix}_metric_result.json").is_file():
        tail = "\n".join((done.stderr or done.stdout).strip().splitlines()[-25:])
        raise RuntimeError(
            f"The evaluator did not finish (exit {done.returncode}). Its last lines:\n{tail}\n"
            f"The whole output is in {work_dir / 'evaluator.log'}."
        )
    return prefix


def headline(prefix):
    """The five overall scores of a finished run, as a one-column table."""
    result = json.loads(Path(f"{prefix}_metric_result.json").read_text())
    values = {name: float(result[e]["all"][m][k]) for name, (e, m, k) in HEADLINE.items()}
    return pd.Series(values, name="score").to_frame()


def per_page(prefix, pages):
    """One row per page of a finished run: what the page is, and its five scores."""

    def read(name):
        return json.loads(Path(f"{prefix}_{name}.json").read_text())

    teds = {}
    for key, value in read("table_per_table_TEDS").items():
        teds.setdefault(re.sub(r"_\[\d+\]$", "", key), []).append(value)
    text, table, order = (read(f"{e}_per_page_edit") for e in ("text_block", "table", "reading_order"))
    rows = []
    for page in pages:
        image = Path(page["page_info"]["image_path"]).name
        attribute = page["page_info"]["page_attribute"]
        tables = teds.get(image, [])
        rows.append(
            {
                "image": image,
                "doc_type": attribute["doc_type"],
                "layout": attribute["layout_id"],
                "tier": attribute["tier"],
                "text_edit": text.get(image),
                "table_teds": sum(t["TEDS"] for t in tables) / len(tables) if tables else None,
                "table_teds_structure": sum(t["TEDS_structure_only"] for t in tables) / len(tables)
                if tables
                else None,
                "table_edit": table.get(image),
                "reading_order_edit": order.get(image),
            }
        )
    return pd.DataFrame(rows).set_index("image")


def matched_lines(prefix, image):
    """Every ground-truth line of one page beside what the evaluator paired it with, worst first."""
    rows = json.loads(Path(f"{prefix}_text_block_result.json").read_text())
    frame = pd.DataFrame(
        [
            {"ground truth": row["gt"], "your output": row["pred"], "edit": row["edit"]}
            for row in rows
            if row["img_id"] == image
        ]
    )
    return frame.sort_values("edit", ascending=False).reset_index(drop=True)


def matched_tables(prefix, image):
    """Each ground-truth table of one page beside the table the evaluator paired it with."""
    rows = json.loads(Path(f"{prefix}_table_result.json").read_text())
    return [(row["gt"], row["pred"]) for row in rows if row["img_id"] == image]


# --- Used only by the self-test in section 4 ---


def pipe_table(table_html):
    """A ground-truth table as a markdown pipe table; one with a spanning cell stays html."""
    if re.search(r"(col|row)span", table_html):
        return table_html
    rows = [
        re.findall(r"<td>(.*?)</td>", row, re.S) for row in re.findall(r"<tr>(.*?)</tr>", table_html, re.S)
    ]
    lines = ["| " + " | ".join(html.unescape(c).replace("|", "\\|") for c in row) + " |" for row in rows]
    lines.insert(1, "|" + "|".join("---" for _ in rows[0]) + "|")
    return "\n".join(lines)


def perfect_markdown(page, tables="html"):
    """A page's ground truth written as the markdown a perfect parser would return."""
    parts = []
    for block in sorted(page["layout_dets"], key=lambda b: b["order"]):
        category = block["category_type"]
        if category == "title":
            parts.append(f"# {block['text']}")
        elif category in ("text_block", "header", "footer"):
            parts.append(block["text"])
        elif category == "table":
            parts.append(pipe_table(block["html"]) if tables == "pipe" else block["html"])
    return "\n\n".join(parts) + "\n"


print(f"Kit: {KIT.name}; sets: {SETS}; evaluator commit: {EVALUATOR_COMMIT}")

## 3. Preflight

Checks the settings: that the evaluator's files are those of the right commit, the work folder, and that
every page of the set has a markdown file. It stops with a list of what to fix if anything is wrong.

In [ ]:
pages = load_pages(SET)
preflight(pages, PREDICTIONS, WORK_DIR)

## 4. Self-test: does the scorer give a perfect answer a perfect score?

Run this once after installing the evaluator, and again if you change the installation. It writes the
set's own ground truth back as markdown, once with html tables and once with pipe tables, and scores
both. Every text and reading-order score must be 0 and every table score 1 (table edit distance 0).
If they are not, the installation is not the one this kit was checked with: stop and fix that before
trusting any score. It takes a few minutes and does not touch your parser's output.

It also shows you exactly the two forms of output the scorer reads: look in
`WORK_DIR/self_test/perfect_html/` and `WORK_DIR/self_test/perfect_pipe/` afterwards.

In [ ]:
self_test = {}
for tables in ("html", "pipe"):
    folder = Path(WORK_DIR) / "self_test" / f"perfect_{tables}"
    folder.mkdir(parents=True, exist_ok=True)
    for page in pages:
        (folder / f"{stem(page)}.md").write_text(perfect_markdown(page, tables))
    prefix = run_evaluator(SET, folder, Path(WORK_DIR) / "self_test" / f"scores_{tables}")
    self_test[f"{tables} tables"] = headline(prefix)["score"]
self_test = pd.DataFrame(self_test)
expected = [0.0, 1.0, 1.0, 0.0, 0.0]
passed = all((self_test[column] == expected).all() for column in self_test)
print("SELF-TEST PASSED" if passed else "SELF-TEST FAILED: do not trust scores from this installation")
self_test

## 5. Score your parser

This is the run that matters. It takes a few minutes for 165 pages.

In [ ]:
prefix = run_evaluator(SET, PREDICTIONS, WORK_DIR)
print(f"Results are in {prefix.parent}")
headline(prefix)

## 6. Scores by document type, layout and tier

`scores` has one row per page. Each overall score above is the mean of its column here, so any grouping
of these rows is consistent with the headline. Bank statements, invoices and receipts differ a great deal
(a statement is nearly all table, a receipt nearly all text), so read them separately.

On the `degraded` set, `tier` is the severity of the photograph: light, moderate or heavy.

In [ ]:
scores = per_page(prefix, pages)
scores.to_csv(Path(WORK_DIR) / "scores_per_page.csv")
metrics = ["text_edit", "table_teds", "table_teds_structure", "table_edit", "reading_order_edit"]
by_type = scores.groupby("doc_type")[metrics].mean()
by_type.loc["all pages"] = scores[metrics].mean()
by_type.round(3)

In [ ]:
group = ["doc_type", "tier"] if scores["tier"].nunique() > 1 else ["doc_type", "layout"]
scores.groupby(group)[metrics].agg("mean").assign(pages=scores.groupby(group).size()).round(3)

## 7. The pages that scored worst

Sorted by text edit distance. Change the column to `table_teds` (ascending) or `reading_order_edit` to
see the worst pages for tables or reading order.

In [ ]:
scores.sort_values("text_edit", ascending=False).head(15).round(3)

## 8. One page, line by line

Set `IMAGE` to any page. The first table shows each ground-truth line beside the text of your output
that the evaluator paired it with, worst first: an `edit` of 0 is an exact match, 1 is a line with no
counterpart. A row whose ground truth is empty is text in your output that matches nothing on the page.
The second cell shows the page's table, ground truth first, then yours as the evaluator read it.

In [ ]:
IMAGE = scores["text_edit"].idxmax()  # or any file name, such as "CASE001_invoice.png"
print(IMAGE, scores.loc[IMAGE].to_dict())
with pd.option_context("display.max_colwidth", 80, "display.max_rows", 200):
    display(matched_lines(prefix, IMAGE))

In [ ]:
from IPython.display import HTML

for truth, yours in matched_tables(prefix, IMAGE):
    display(HTML("<b>Ground truth</b>" + truth.replace("<table>", "<table border='1'>")))
    display(
        HTML(
            "<b>Your output</b>"
            + (yours or "<i>no table matched</i>").replace("<table>", "<table border='1'>")
        )
    )

## 9. Comparing two sets

The three clean sets hold the same 165 documents under the same file names, so two runs pair page by
page. Score each set into its own work folder (change `SET`, `PREDICTIONS` and `WORK_DIR` and rerun
sections 3, 5 and 6), then compare the two `scores_per_page.csv` files. For example, what the
distractors cost your parser:

```python
plain = pd.read_csv("/path/to/plain/scores_per_page.csv", index_col="image")
distractor = pd.read_csv("/path/to/distractor/scores_per_page.csv", index_col="image")
difference = (distractor[metrics] - plain[metrics]).assign(doc_type=plain["doc_type"])
difference.groupby("doc_type").mean().round(3)
```

For the photographed receipts, pair each `degraded` page with the receipt it is a photograph of: strip
the `_v1`, `_v2` or `_v3` from its name and look that receipt up in the `distractor` scores. Compare
receipts with receipts only: the degraded set holds no statements or invoices.